# Behaviour Prediction Notebook

End-to-end experiment notebook with leakage-safe preprocessing and repeated cross-validation. The goal is to measure predictive benefits associated with different types of encoding of brain lesion images. The encoded information is deployed together with non-lesion factors, to predict patients' language outcomes after stroke. The patient data are naturally confidential so cannot be shared publicly.

The notebook expects a cross-sectional dataset, and will treat each row as its own patient.

The baseline is an XGBoost model with tabular lesion load variables, representing the % of a series of anatomically defined regions, that the patient's lesion destroys. Coupled with a few non-lesion factors such as age at stroke onset and pre-stroke handedness, this model achieves SOA performance with these data.

There are then three different CNN models, which each represent the lesion data differently (but also receive the non-lesion factors as inputs). In this case, the lesion images are derived from an algorithmic segmentation process, which calculates abnormality of the patient's brain at the voxel level, relative to an age matched distribution of neurologically normal controls.

This 'fuzzy' lesion image does indeed drive better predictions. We are also interested in whether the benefit accrues from continuous lesion information either inside or outside what would otherwise be considered the (binary) lesion boundary.

## Environment Setup

In [ ]:
!pip install -q h5py scikit-learn xgboost tensorflow pandas seaborn matplotlib

## Imports and Global Configuration

In [ ]:
import os, gc, time, random, warnings, h5py
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf
import xgboost as xgb

from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from tensorflow.keras.layers import Input,Dense,Conv3D,MaxPooling3D,GlobalAveragePooling3D,Concatenate,Dropout
from tensorflow.keras.models import Model

warnings.filterwarnings("ignore")
SEED=42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

BEHAVIOUR_COLUMNS=[5,6,8,9,10,7,15,18,23,27,28,33,37]
N_REPEATS=5
N_FOLDS=5
MAX_EPOCHS=1000
BATCH_SIZE=64
LEARNING_RATE=1e-6
PATIENCE=10
OUTPUT_DIR='DATA_FOLDER'
os.makedirs(OUTPUT_DIR,exist_ok=True)

## Helper Functions

In [ ]:
def load_mat_array(path):
    with h5py.File(path,"r") as f:
        return np.array(f["im_array"][:])

def preprocess_fold(Xtr,Xte,Dtr,Dte,ytr,yte):
    sx,sd,sy=StandardScaler(),StandardScaler(),StandardScaler()
    Xtr=sx.fit_transform(Xtr); Xte=sx.transform(Xte)
    Dtr=sd.fit_transform(Dtr); Dte=sd.transform(Dte)
    ytr=sy.fit_transform(ytr.reshape(-1,1)).ravel()
    yte=sy.transform(yte.reshape(-1,1)).ravel()
    return Xtr,Xte,Dtr,Dte,ytr,yte

def scale_images(train_imgs,test_imgs):
    m=np.mean(train_imgs); s=np.std(train_imgs)
    return ((train_imgs-m)/(s+1e-8)).astype("float32"),((test_imgs-m)/(s+1e-8)).astype("float32")

## Model Definitions

In [ ]:
def create_cnn(n_demographic_features,image_shape):
    demo_in=Input(shape=(n_demographic_features,))
    img_in=Input(shape=image_shape)

    d=Dense(32,activation="relu")(demo_in)
    d=Dense(16,activation="relu")(d)

    x=Conv3D(16,3,padding="same",activation="relu")(img_in)
    x=MaxPooling3D(2)(x)
    x=Conv3D(32,3,padding="same",activation="relu")(x)
    x=MaxPooling3D(2)(x)
    x=Conv3D(64,3,padding="same",activation="relu")(x)
    x=GlobalAveragePooling3D()(x)
    x=Dense(64,activation="relu")(x)

    z=Concatenate()([d,x])
    z=Dense(64,activation="relu")(z)
    #z=Dropout(0.3)(z)
    z=Dense(32,activation="relu")(z)
    out=Dense(1)(z)

    model=Model([demo_in,img_in],out)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),loss="mae")
    return model

def train_baseline(Xtr,ytr,Xte,yte):
    model=xgb.XGBRegressor(n_estimators=300,max_depth=4,learning_rate=0.05,random_state=SEED)
    model.fit(Xtr,ytr)
    pred=model.predict(Xte)
    return mean_squared_error(yte,pred)

def train_cnn(Dtr,Itr,ytr,Dte,Ite,yte,logdir=None):
    print("line 1")
    model=create_cnn(Dtr.shape[1],Itr.shape[1:])
    print("line 2")
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=PATIENCE,restore_best_weights=True)]
    print("line 3")
    if logdir: callbacks.append(tf.keras.callbacks.TensorBoard(log_dir=logdir))
    print("line 4")
    hist=model.fit([Dtr,Itr],ytr,validation_data=([Dte,Ite],yte),epochs=MAX_EPOCHS,batch_size=BATCH_SIZE,verbose=1,callbacks=callbacks)
    print("line 5")
    mse=float(model.evaluate([Dte,Ite],yte,verbose=0))
    print("line 6")
    best=min(hist.history["val_loss"])
    print("line 7")
    tf.keras.backend.clear_session(); del model; gc.collect()
    return mse,best

## Data Loading

In [ ]:
image_fuzzy=load_mat_array(f"{OUTPUT_DIR}/ims_fuz.mat").astype("float32")
image_binary=load_mat_array(f"{OUTPUT_DIR}/ims_bin.mat").astype("float32")
image_inside=load_mat_array(f"{OUTPUT_DIR}/ims_fuz_in.mat").astype("float32")
image_outside=load_mat_array(f"{OUTPUT_DIR}/ims_fuz_out.mat").astype("float32")
predictors=np.genfromtxt(f"{OUTPUT_DIR}/predictors.csv",delimiter=",")
outcomes=np.genfromtxt(f"{OUTPUT_DIR}/behavioural.csv",delimiter=",")

image_fuzzy = np.squeeze(image_fuzzy,axis=1)
image_binary = np.squeeze(image_binary,axis=1)
image_inside = np.squeeze(image_inside,axis=1)
image_outside = np.squeeze(image_outside,axis=1)

image_fuzzy = image_fuzzy[..., np.newaxis]
image_binary = image_binary[..., np.newaxis]
image_inside = image_inside[..., np.newaxis]
image_outside = image_outside[..., np.newaxis]


## Data Cleaning

In [ ]:
valid=~(np.any(np.isnan(predictors),axis=1)|np.any(np.isnan(outcomes),axis=1))
predictors=predictors[valid]
outcomes=outcomes[valid]
image_fuzzy=image_fuzzy[valid]
image_binary=image_binary[valid]
image_inside=image_inside[valid]
image_outside=image_outside[valid]
demographics=predictors[:,:5]
print("Subjects:",len(predictors))
print(image_fuzzy.shape)
print(image_fuzzy[0].shape)

## Main Experiment

In [ ]:
results=[]
experiment_start=time.time()
started_folds = 0
TOTAL_FOLDS = N_REPEATS * N_FOLDS * len(BEHAVIOUR_COLUMNS)

for behaviour in BEHAVIOUR_COLUMNS:
    for repeat in range(N_REPEATS):
        cv=KFold(n_splits=N_FOLDS,shuffle=True,random_state=repeat)

        for fold,(train_idx,test_idx) in enumerate(cv.split(predictors)):
            started_folds += 1
            progress = (started_folds / TOTAL_FOLDS) * 100
            print(
                f"\nProgress: "
                f"{started_folds}/{TOTAL_FOLDS} "
                f"({progress:.1f}%)"
            )
            try:
                Xtr_raw,Xte_raw=predictors[train_idx],predictors[test_idx]
                print("line 1")
                Dtr_raw,Dte_raw=demographics[train_idx],demographics[test_idx]
                print("line 2")
                ytr_raw=outcomes[train_idx,behaviour]
                print("line 3")
                yte_raw=outcomes[test_idx,behaviour]
                print("line 4")

                Xtr,Xte,Dtr,Dte,ytr,yte=preprocess_fold(Xtr_raw,Xte_raw,Dtr_raw,Dte_raw,ytr_raw,yte_raw)
                print("line 5")

                fuzzy_tr,fuzzy_te=scale_images(image_fuzzy[train_idx],image_fuzzy[test_idx])
                print("line 6")
                binary_tr,binary_te=scale_images(image_binary[train_idx],image_binary[test_idx])
                print("line 7")
                inside_tr,inside_te=scale_images(image_inside[train_idx],image_inside[test_idx])
                print("line 8")
                outside_tr,outside_te=scale_images(image_outside[train_idx],image_outside[test_idx])
                print("line 9")

                baseline=train_baseline(Xtr,ytr,Xte,yte)
                print("line 10")
                fuzzy_mse,fuzzy_best=train_cnn(Dtr,fuzzy_tr,ytr,Dte,fuzzy_te,yte)
                print("line 11")
                binary_mse,binary_best=train_cnn(Dtr,binary_tr,ytr,Dte,binary_te,yte)
                print("line 13")
                inside_mse,inside_best=train_cnn(Dtr,inside_tr,ytr,Dte,inside_te,yte)
                print("line 14")
                outside_mse,outside_best=train_cnn(Dtr,outside_tr,ytr,Dte,outside_te,yte)
                print("line 15")

                results.append(dict(behaviour=behaviour,repeat=repeat,fold=fold,baseline_mse=baseline,cnn_fuzzy_mse=fuzzy_mse,cnn_binary_mse=binary_mse,cnn_inside_mse=inside_mse,cnn_outside_mse=outside_mse,best_val_fuzzy=fuzzy_best))

                if len(results)%10==0:
                    pd.DataFrame(results).to_csv(f"{OUTPUT_DIR}/model_results_checkpoint.csv",index=False)

            except Exception as e:
                print("Failed",behaviour,repeat,fold,e)

results_df=pd.DataFrame(results)
results_df.to_csv(f"{OUTPUT_DIR}/model_results.csv",index=False)
print("Hours:",round((time.time()-experiment_start)/3600,2))

## Summary Statistics and Confidence Intervals

In [ ]:
results_df=pd.read_csv(f"{OUTPUT_DIR}/model_results.csv")
summary=results_df.groupby("behaviour")[["baseline_mse","cnn_fuzzy_mse","cnn_binary_mse","cnn_inside_mse","cnn_outside_mse"]].agg(["mean","std","count"])
summary.to_csv(f"{OUTPUT_DIR}/summary_statistics.csv")
summary.head()

## Visualisations

In [ ]:
plt.figure(figsize=(12,6))
plot_df=results_df.melt(id_vars=["behaviour"],value_vars=["baseline_mse","cnn_fuzzy_mse","cnn_binary_mse","cnn_inside_mse","cnn_outside_mse"])
sns.boxplot(data=plot_df,x="variable",y="value")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/model_comparison.png")
plt.show()

## Behaviour-wise Performance

In [ ]:
mean_perf=results_df.groupby("behaviour")[["baseline_mse","cnn_fuzzy_mse","cnn_binary_mse","cnn_inside_mse","cnn_outside_mse"]].mean()
mean_perf.plot(figsize=(12,6),marker="o")
plt.ylabel("Mean MSE")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/behaviour_performance.png")
plt.show()